# Notebook 04 — Benchmark Validation: Farde et al. 1992

Validates the D2 occupancy model against the landmark PET study:

> Farde L, Nordstrom AL, Wiesel FA, Pauli S, Halldin C, Sedvall G (1992).  
> Positron emission tomographic analysis of central D1 and D2 dopamine  
> receptor occupancy in patients treated with classical neuroleptics and  
> clozapine. **Arch Gen Psychiatry 49:538–544.** PMID: 1352677.

**Key finding to reproduce**: Haloperidol 4–12 mg/day → 75–89% D2 occupancy  
**Clinical thresholds**: 65–80% therapeutic window; >80% EPS risk


In [ ]:
import json
import pathlib
import numpy as np
import matplotlib.pyplot as plt
import pandas as pd
from scipy.optimize import curve_fit
%matplotlib inline

from neuropharm_sim.receptor_dynamics import antagonist_d2_occupancy, dose_to_brain_concentration
from neuropharm_sim.drug_library import get_drug
from neuropharm_sim.visualization import plot_benchmark_validation

## 1. Load Farde 1992 data

In [ ]:
data_path = pathlib.Path("..") / "data" / "farde1992_haloperidol.json"
with open(data_path) as f:
    farde_raw = json.load(f)

source = farde_raw["_source"]
print(f"Source: {source['citation']}")
print(f"PMID  : {source['pmid']}")
print(f"DOI   : {source['doi']}")
print(f"Method: {source['radioligand']} {source['modality']}")
print()

dp = farde_raw["haloperidol_dose_occupancy"]["data_points"]
farde_doses = np.array([pt["dose_mg"] for pt in dp])
farde_occ   = np.array([pt["d2_occupancy_pct"] for pt in dp]) / 100.0

thresholds = farde_raw["haloperidol_dose_occupancy"]["clinical_thresholds"]

print("Dose (mg/day) | D2 occupancy")
for d, o in zip(farde_doses, farde_occ):
    print(f"  {d:5.1f}       | {o*100:.1f}%")

## 2. Model prediction across dose range

In [ ]:
haldol = get_drug("haloperidol")
doses_mg = np.linspace(0.1, 15, 200)

brain_nm = dose_to_brain_concentration(
    doses_mg,
    bioavailability=haldol.bioavailability,
    volume_dist_l=haldol.volume_dist_l,
    molecular_weight=haldol.molecular_weight,
    brain_plasma_ratio=haldol.brain_plasma_ratio,
    free_fraction=haldol.free_fraction_plasma,
)

simulated_occ = antagonist_d2_occupancy(brain_nm, ki_nm=haldol.ki_d2_nm)

print(f"At 5 mg: brain conc = {float(dose_to_brain_concentration(5.0, haldol.bioavailability, haldol.volume_dist_l, haldol.molecular_weight, haldol.brain_plasma_ratio, haldol.free_fraction_plasma)):.3f} nM")
print(f"At 5 mg: D2 occupancy = {float(antagonist_d2_occupancy(dose_to_brain_concentration(5.0, haldol.bioavailability, haldol.volume_dist_l, haldol.molecular_weight, haldol.brain_plasma_ratio, haldol.free_fraction_plasma), ki_nm=haldol.ki_d2_nm))*100:.1f}%")

## 3. Benchmark plot

In [ ]:
fig, ax = plot_benchmark_validation(
    doses_mg=doses_mg,
    simulated_occ=simulated_occ,
    farde_doses=farde_doses,
    farde_occ=farde_occ,
)
plt.show()

## 4. Residual analysis

In [ ]:
# Predicted occupancy at each Farde dose
c_at_farde = dose_to_brain_concentration(
    farde_doses,
    bioavailability=haldol.bioavailability,
    volume_dist_l=haldol.volume_dist_l,
    molecular_weight=haldol.molecular_weight,
    brain_plasma_ratio=haldol.brain_plasma_ratio,
    free_fraction=haldol.free_fraction_plasma,
)
pred_occ = antagonist_d2_occupancy(c_at_farde, ki_nm=haldol.ki_d2_nm)

residuals = (pred_occ - farde_occ) * 100  # percentage points

rows = []
for d, obs, pred, res in zip(farde_doses, farde_occ*100, pred_occ*100, residuals):
    rows.append({
        "Dose (mg/day)": d,
        "Farde 1992 (%)": f"{obs:.1f}",
        "Model predicted (%)": f"{float(pred):.1f}",
        "Residual (pp)": f"{float(res):+.1f}",
    })

df = pd.DataFrame(rows)
print(df.to_string(index=False))
print(f"\nRMSE: {np.sqrt(np.mean(residuals**2)):.2f} percentage points")
print(f"MAE : {np.mean(np.abs(residuals)):.2f} percentage points")

## 5. Clinical threshold verification

Confirm the model identifies the therapeutic window and EPS risk doses.

In [ ]:
therapeutic_lo = thresholds["antipsychotic_effect_lower_pct"] / 100
therapeutic_hi = thresholds["antipsychotic_effect_upper_pct"] / 100
eps_threshold  = thresholds["eps_risk_pct"] / 100

# Find doses at threshold crossings
idx_lo  = np.argmin(np.abs(simulated_occ - therapeutic_lo))
idx_hi  = np.argmin(np.abs(simulated_occ - therapeutic_hi))
idx_eps = np.argmin(np.abs(simulated_occ - eps_threshold))

print("Clinical threshold doses (model prediction):")
print(f"  Therapeutic lower ({therapeutic_lo*100:.0f}%) : {doses_mg[idx_lo]:.2f} mg/day")
print(f"  Therapeutic upper ({therapeutic_hi*100:.0f}%) : {doses_mg[idx_hi]:.2f} mg/day")
print(f"  EPS risk (>{eps_threshold*100:.0f}%)          : {doses_mg[idx_eps]:.2f} mg/day")
print()
print("Reference (Kapur & Seeman 2001; Farde 1992):")
print("  Therapeutic window: ~1–5 mg/day (clinical practice)")
print("  EPS commonly at   : >5 mg/day in sensitive patients")

## 6. Compare model with empirical Hill fit

In [ ]:
def emax_model(dose, ed50, n):
    return dose**n / (ed50**n + dose**n)

popt, _ = curve_fit(emax_model, farde_doses, farde_occ,
                     p0=[2.0, 1.0], bounds=([0, 0.1], [100, 5]))
ed50_fit, n_fit = popt

empirical_fit = emax_model(doses_mg, *popt)

fig, ax = plt.subplots(figsize=(7, 5))
ax.plot(doses_mg, simulated_occ * 100, lw=2.2, color="#2166ac", label="mechanistic model")
ax.plot(doses_mg, empirical_fit * 100, lw=1.8, ls="--", color="#762a83",
        label=f"empirical Hill fit (ED50={ed50_fit:.2f} mg, n={n_fit:.2f})")
ax.scatter(farde_doses, farde_occ * 100, s=70, zorder=5, color="#d6604d",
           edgecolors="white", linewidths=1, label="Farde et al. 1992")
ax.set_xlabel("Haloperidol dose (mg/day)")
ax.set_ylabel("D2 receptor occupancy (%)")
ax.set_title("Mechanistic vs Empirical D2 Occupancy Model", fontweight="bold")
ax.set_ylim(0, 105)
ax.legend(fontsize=9.5)
plt.tight_layout()
plt.show()

print(f"Empirical Hill fit: ED50 = {ed50_fit:.2f} mg/day, n = {n_fit:.2f}")